# PROJECT 9: Customer Segmentation using K-Means
**Domain:** Unsupervised Learning – Clustering
**Difficulty:** Beginner to Intermediate

---

## A. Problem Statement
Retailers need to understand their customer base to optimize marketing strategies. A generic approach to all customers leads to wasted resources and lower conversion rates.

## B. Project Objective
Identify different customer groups using K-Means clustering and provide business insights for targeted marketing and personalized promotions.

## C. Dataset Description
The `Mall_Customers.csv` dataset contains:
- **CustomerID:** Unique ID assigned to the customer
- **Gender:** Gender of the customer
- **Age:** Age of the customer
- **Annual Income (k$):** Annual Income of the customer
- **Spending Score (1-100):** Score assigned by the mall based on customer behavior and spending nature

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## D. Exploratory Data Analysis (EDA)
Performing shape(), info(), describe(), missing values, duplicate check, and statistical summary.

In [ ]:
df = pd.read_csv('dataset/Mall_Customers.csv')
print(f"Shape: {df.shape}")
print("-" * 50)
df.info()
print("-" * 50)
print(f"Missing Values:\n{df.isnull().sum()}")
print("-" * 50)
print(f"Duplicates: {df.duplicated().sum()}")
print("-" * 50)
display(df.describe())

### Visualizations
Histograms, count plots, correlation heatmap, pairplot, distribution plots, and scatter plots.

In [ ]:
# 1. Histograms for numerical columns
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Age Distribution')
sns.histplot(df['Annual Income (k$)'], bins=20, kde=True, ax=axes[1], color='lightgreen')
axes[1].set_title('Annual Income Distribution')
sns.histplot(df['Spending Score (1-100)'], bins=20, kde=True, ax=axes[2], color='salmon')
axes[2].set_title('Spending Score Distribution')
plt.show()

# 2. Count plot for Gender
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Gender', palette='pastel')
plt.title('Gender Count')
plt.show()

# 3. Correlation Heatmap
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

# 4. Pairplot
sns.pairplot(df.drop('CustomerID', axis=1), hue='Gender', palette='Set2')
plt.show()

# 5. Scatter Plot (Income vs Spending)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Gender', palette='Set1', s=100)
plt.title('Income vs Spending Score')
plt.show()

## E. Data Cleaning & Preprocessing
- Handling missing values and duplicates (None found in EDA).
- Encoding categorical variables.
- Normalization using StandardScaler (Critical for K-Means).

In [ ]:
# Drop CustomerID
data = df.drop(['CustomerID'], axis=1).copy()

# Encode Gender
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

print("Data after encoding:")
display(data.head())

## F. Feature Engineering
Creating derived features to potentially help segmentation:
- **Spending_per_Age**: Spending relative to age.
- **Income_to_Spending_Ratio**: Income relative to spending.
- **Loyalty_Score**: A derived metric assuming higher age and higher spending relates to loyalty.

In [ ]:
data['Spending_per_Age'] = data['Spending Score (1-100)'] / data['Age']
data['Income_to_Spending_Ratio'] = data['Annual Income (k$)'] / (data['Spending Score (1-100)'] + 1e-5)
data['Loyalty_Score'] = (data['Age'] * 0.2) + (data['Spending Score (1-100)'] * 0.8)

display(data.head())

# Selecting core numerical features for clustering
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = data[features]

# Normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Features standardized.")

## G. Model Building (K-Means)
Determine optimal K using the Elbow Method and Silhouette Score.

In [ ]:
wcss = []
silhouette_scores = []
K_RANGE = range(2, 11)

for k in K_RANGE:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plotting Elbow and Silhouette
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(K_RANGE, wcss, marker='o', linestyle='--', color='b')
axes[0].set_title('Elbow Method (WCSS)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('WCSS')

axes[1].plot(K_RANGE, silhouette_scores, marker='s', linestyle='-', color='g')
axes[1].set_title('Silhouette Score Analysis')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

In [ ]:
# Based on the plots, optimal K is usually 5
optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Add cluster labels back to dataframe
df['Cluster'] = cluster_labels
data['Cluster'] = cluster_labels
print("Clustering completed and labels assigned.")

## H. Cluster Evaluation
Calculating Inertia, Silhouette Score, and Davies-Bouldin Index.
- **Inertia (WCSS):** Measures within-cluster sum of squares (lower is better).
- **Silhouette Score:** Measures how similar an object is to its own cluster compared to others (1 is best, -1 is worst).
- **Davies-Bouldin Index:** Measures average similarity between clusters (lower is better, signifies well-separated clusters).

In [ ]:
inertia = kmeans_final.inertia_
sil_score = silhouette_score(X_scaled, cluster_labels)
db_index = davies_bouldin_score(X_scaled, cluster_labels)

print(f"Inertia (WCSS): {inertia:.2f}")
print(f"Silhouette Score: {sil_score:.4f}")
print(f"Davies-Bouldin Index: {db_index:.4f}")

## Visualizing the Clusters
2D Scatter Plot, 3D Scatter Plot, PCA visualization, and Cluster Distribution.

In [ ]:
# 1. 2D Scatter Plot (Income vs Spending)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', palette='viridis', s=100)
# Plot centroids
centroids_2d = scaler.inverse_transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_2d[:, 1], centroids_2d[:, 2], s=300, c='red', marker='X', label='Centroids')
plt.title('Clusters: Income vs Spending Score')
plt.legend()
plt.show()

In [ ]:
# 2. 3D Scatter Plot using Plotly
fig = px.scatter_3d(df, x='Age', y='Annual Income (k$)', z='Spending Score (1-100)',
                    color='Cluster', opacity=0.8, title='3D Cluster Visualization',
                    color_continuous_scale=px.colors.sequential.Viridis)
fig.show()

In [ ]:
# 3. PCA-based visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=cluster_labels, palette='tab10', s=100)
plt.title('PCA - 2D Projection of Clusters')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

In [ ]:
# 4. Cluster Distribution Chart
plt.figure(figsize=(6, 5))
sns.countplot(data=df, x='Cluster', palette='Set3')
plt.title('Number of Customers per Cluster')
plt.show()

## I. Business Insights & Cluster Analysis
Understanding the characteristics of each cluster based on average values.

In [ ]:
cluster_analysis = df.groupby('Cluster').agg({
    'Age': 'mean',
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'Size'}).reset_index()

display(cluster_analysis.round(2))

### Interpretation Example:
- **Cluster 0:** Medium income + loyal customers.
- **Cluster 1:** High income + high spending (Premium customers).
- **Cluster 2:** Low income + low spending (Budget customers).
- **Cluster 3:** Young high spenders.
- **Cluster 4:** Occasional shoppers.
*(Note: Cluster IDs map differently on each run)*

## J. Conclusion
1. **Which customer segment is most valuable?** The Premium customers (High Income, High Spending) are the most valuable for high-margin products.
2. **Which segment requires retention efforts?** The Young High Spenders and Occasional Shoppers need targeted campaigns to build brand loyalty.
3. **How can segmentation improve business revenue?** By preventing generic marketing. Ad spend can be optimized by targeting only relevant clusters.
4. **How can personalized marketing use these clusters?** Sending luxury brand promos to Premium customers and clearance sale emails to Budget customers.

## K. Future Scope
- Incorporating temporal data (frequency of visits, time spent in mall).
- Using Density-Based Clustering (DBSCAN) or Hierarchical Clustering for comparison.
- Deploying the model via a Web App (Flask/Streamlit) for real-time customer segmentation.